<a href="https://colab.research.google.com/github/KlyffHanger/TinyML/blob/main/ConvertingModel_TF_2_TFL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!pip install tensorflow
#!pip install tensorflow==2.20.0

import tensorflow as tf
print(tf.__version__)

# if tf.__version__ != "2.14.0":
#     print(f"Current TensorFlow version: {tf.__version__}, switching to 2.14.0")

#     # Uninstall current TensorFlow version
#     !pip uninstall -y tensorflow

#     # Install TensorFlow 2.10
#     !pip install numpy==1.26 --force-reinstall
#     !pip install tensorflow==2.14.0


#     # After installation, restart runtime
#     print("TensorFlow 2.14.0 installed.")
#     print("Please click on the Runtime > Restart session and run all.")
# else:
#     print("TensorFlow 2.14.0 is already installed.")

2.19.0


In [2]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [3]:
print(f"Reconfirming the version ${tf.__version__}")

l0 = Dense(units=1, input_shape=[1])
model = Sequential([l0])
model.compile(optimizer='sgd', loss='mean_squared_error')

xs = np.array([-1.0, 0.0, 1.0, 2.0, 3.0, 4.0], dtype=float)
ys = np.array([-3.0, -1.0, 1.0, 3.0, 5.0, 7.0], dtype=float)

model.fit(xs, ys, epochs=500)

print(model.predict(np.array([10.0])))
print("Here is what I learned: {}".format(l0.get_weights()))

Reconfirming the version $2.19.0


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 415ms/step - loss: 12.1410
Epoch 2/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - loss: 9.7758
Epoch 3/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - loss: 7.9105
Epoch 4/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - loss: 6.4384
Epoch 5/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - loss: 5.2758
Epoch 6/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 4.3568
Epoch 7/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 3.6296
Epoch 8/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 3.0533
Epoch 9/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 2.5958
Epoch 10/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 2.2319
Epoch 11/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 1.9417
Epoch 12/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 1.7096
Epoch 13/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 1.5232
Epoch 14/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 1.3730
Epoch 15/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - loss: 1.2511
Epoch 16/500
1

In [4]:
export_dir = 'saved_model/1'
model.export(export_dir)

Saved artifact at 'saved_model/1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134560143739024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134560143740176: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [5]:
#!pip install ai_edge_litert

In [6]:
# Convert the model.
converter = tf.lite.TFLiteConverter.from_saved_model(export_dir)
tflite_model = converter.convert()

In [7]:
import pathlib
tflite_model_file = pathlib.Path('model.tflite')
tflite_model_file.write_bytes(tflite_model)

1104

In [8]:
#Load TFLite model and allocate tensors.
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

# Get input and output tensors.
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print(input_details)
print(output_details)

[{'name': 'serving_default_keras_tensor:0', 'index': 0, 'shape': array([1, 1], dtype=int32), 'shape_signature': array([-1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
[{'name': 'StatefulPartitionedCall_1:0', 'index': 3, 'shape': array([1, 1], dtype=int32), 'shape_signature': array([-1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [9]:
# from ai_edge_litert.interpreter import LiteRTInterpreter

# #from ai_edge_litert.interpreter import Interpreter
# #interpreter = Interpreter(model_path=args.model_file)

# # Load TFLite model and allocate tensors using LiteRT.
# # Note: model_content is preferred for in-memory models.
# litert_interpreter = LiteRTInterpreter(model_content=tflite_model)
# litert_interpreter.allocate_tensors()

# # Get input and output tensors details for LiteRT
# litert_input_details = litert_interpreter.get_input_details()
# litert_output_details = litert_interpreter.get_output_details()

# print("LiteRT Input Details:")
# print(litert_input_details)
# print("\nLiteRT Output Details:")
# print(litert_output_details)

In [10]:
# import numpy as np

# # Example input for LiteRT
# to_predict_litert = np.array([[10.0]], dtype=np.float32)

# # Set the tensor, invoke, and get results using LiteRT
# litert_interpreter.set_tensor(litert_input_details[0]['index'], to_predict_litert)
# litert_interpreter.invoke()
# litert_results = litert_interpreter.get_tensor(litert_output_details[0]['index'])

# print(f"Input: {to_predict_litert[0][0]}, LiteRT Prediction: {litert_results[0][0]:.4f}")

In [11]:
to_predict = np.array([[10.0]], dtype=np.float32)
print(to_predict)
interpreter.set_tensor(input_details[0]['index'], to_predict)
interpreter.invoke()
tflite_results = interpreter.get_tensor(output_details[0]['index'])
print(tflite_results)

[[10.]]
[[18.98227]]


In [14]:
import os

saved_model_size = sum(os.path.getsize(os.path.join(dirpath, filename)) for dirpath, dirnames, filenames in os.walk(export_dir) for filename in filenames)
print(f"SavedModel directory size: {saved_model_size / (1024):.2f} KB")

tflite_model_size = os.path.getsize('model.tflite')
print(f"TFLite model file size: {tflite_model_size / (1024):.2f} KB")

SavedModel directory size: 22.38 KB
TFLite model file size: 1.08 KB
